# Launch forecast: MLflow LightGBM model
Runs next to `run_forecast` (ai_forecast) so the app can compare both. Same series rules: history up to the last complete Monday-week, zero-filled; announced launches after the cutoff are a floor; 12 weeks for every client; clients with < 8 weeks of history get the 8-week mean (0 without history). The global LightGBM model is retrained each run, logged to MLflow, and its forecast is written to `gold.launch_forecasts_mlflow`. See `forecast_exploration` for the backtest.

In [0]:
%pip install -q lightgbm
dbutils.library.restartPython()

In [0]:
import datetime as dt, warnings
import numpy as np, pandas as pd
import lightgbm as lgb, mlflow
from mlflow.models import infer_signature
warnings.filterwarnings("ignore")

dbutils.widgets.text("catalog", "patricia_aws_serverless_catalog")
CATALOG = dbutils.widgets.get("catalog")
H, MIN_HISTORY = 12, 8
EXPERIMENT = "/Users/patricia.ferreiro@databricks.com/launch-forecast-comparison"

weekly = spark.sql(f"""
  SELECT client_id, CAST(DATE_TRUNC('week', launch_date) AS DATE) AS ds, COUNT(*) AS y
  FROM {CATALOG}.silver.product_launches
  WHERE launch_date IS NOT NULL AND client_id IS NOT NULL GROUP BY 1, 2""").toPandas()
weekly["ds"] = pd.to_datetime(weekly["ds"])
TODAY = pd.Timestamp(dt.date.today())
CUTOFF = TODAY - pd.Timedelta(days=TODAY.weekday()) - pd.Timedelta(weeks=1)  # last complete Monday-week
observed, scheduled = weekly[weekly.ds <= CUTOFF], weekly[weekly.ds > CUTOFF]

series = []
for c, g in observed.groupby("client_id"):
    idx = pd.date_range(g.ds.min(), CUTOFF, freq="W-MON")
    series.append(pd.DataFrame({"client_id": c, "ds": idx, "y": g.set_index("ds")["y"].reindex(idx).fillna(0).astype(float).values}))
series = pd.concat(series, ignore_index=True)
history_weeks = series.groupby("client_id").size()
eligible = sorted(history_weeks[history_weeks >= MIN_HISTORY].index)
clients = sorted(weekly.client_id.unique())
future_weeks = pd.date_range(CUTOFF + pd.Timedelta(weeks=1), periods=H, freq="W-MON")
print("cutoff", CUTOFF.date(), "| clients", clients, "| eligible for the model", eligible)

In [0]:
LAGS = [12, 13, 14, 26, 52]  # every lag >= horizon, so future rows only need known history
FEATURES = [f"lag_{l}" for l in LAGS] + ["roll_mean_12_19", "weekofyear", "month", "client"]
LGB_PARAMS = dict(n_estimators=200, learning_rate=0.05, num_leaves=7, min_child_samples=5, verbose=-1, random_state=42)

def make_features(frame, categories):
    frame, hist = frame[["client_id", "ds", "y"]].copy(), frame.dropna(subset=["y"])
    for l in LAGS:
        lagged = hist[["client_id", "ds", "y"]].rename(columns={"y": f"lag_{l}"})
        lagged["ds"] = lagged["ds"] + pd.Timedelta(weeks=l)
        frame = frame.merge(lagged, on=["client_id", "ds"], how="left")
    roll = []
    for c, g in hist.groupby("client_id"):
        r = g.set_index("ds")["y"].asfreq("W-MON").rolling(8, min_periods=4).mean()
        r.index = r.index + pd.Timedelta(weeks=12)  # mean of weeks t-19..t-12
        roll.append(pd.DataFrame({"client_id": c, "ds": r.index, "roll_mean_12_19": r.values}))
    frame = frame.merge(pd.concat(roll), on=["client_id", "ds"], how="left")
    frame["weekofyear"], frame["month"] = frame.ds.dt.isocalendar().week.astype(int), frame.ds.dt.month
    frame["client"] = pd.Categorical(frame.client_id, categories=categories)
    return frame

class LaunchForecaster(mlflow.pyfunc.PythonModel):
    """Takes future rows (client_id, ds = Monday of the week); returns forecast, forecast_lower, forecast_upper."""
    def __init__(self, models, history, categories):
        self.models, self.categories = models, categories
        self.history = history[["client_id", "ds", "y"]].sort_values("ds").groupby("client_id").tail(max(LAGS) + 8)
    def predict(self, context, model_input, params=None):
        future = model_input[["client_id", "ds"]].copy(); future["ds"] = pd.to_datetime(future["ds"])
        frame = make_features(pd.concat([self.history, future.assign(y=np.nan)], ignore_index=True), self.categories)
        rows = frame[frame.y.isna()].copy()
        f, lo, hi = (np.clip(self.models[k].predict(rows[FEATURES]), 0, None) for k in ("mean", 0.05, 0.95))
        rows = rows[["client_id", "ds"]].assign(forecast=f, forecast_lower=np.minimum(lo, f), forecast_upper=np.maximum(hi, f))
        return future.merge(rows, on=["client_id", "ds"], how="left")

train = series[series.client_id.isin(eligible)]
feats = make_features(train, eligible).dropna(subset=["lag_12"])
models = {q: lgb.LGBMRegressor(objective="quantile", alpha=q, **LGB_PARAMS).fit(feats[FEATURES], feats["y"]) for q in (0.05, 0.95)}
models["mean"] = lgb.LGBMRegressor(objective="regression", **LGB_PARAMS).fit(feats[FEATURES], feats["y"])
model = LaunchForecaster(models, train, eligible)
future = pd.DataFrame([(c, t) for c in eligible for t in future_weeks], columns=["client_id", "ds"])
ml = model.predict(None, future)
assert ml["forecast"].notna().all(), "model returned NULL forecasts"
print("training rows:", len(feats))

In [0]:
mlflow.set_experiment(EXPERIMENT)
with mlflow.start_run(run_name=f"forecast_mlflow {CUTOFF.date()}") as run:
    mlflow.set_tags({"project": "product-launch-forecast", "method": "lightgbm_global", "purpose": "production forecast"})
    mlflow.log_params({**{f"lgb_{k}": v for k, v in LGB_PARAMS.items()}, "lags": str(LAGS), "cutoff": str(CUTOFF.date()),
                       "horizon_weeks": H, "clients": ",".join(eligible), "n_train": len(feats)})
    MODEL_URI = mlflow.pyfunc.log_model(name="model", python_model=model, signature=infer_signature(future, ml),
                                        input_example=future.head(5), extra_pip_requirements=[f"lightgbm=={lgb.__version__}"]).model_uri
    RUN_ID = run.info.run_id
print("run", RUN_ID, "| model", MODEL_URI)

In [0]:
grid = pd.DataFrame([(c, t) for c in clients for t in future_weeks], columns=["client_id", "ds"])
base = series.sort_values("ds").groupby("client_id").tail(8).groupby("client_id")["y"].agg(mean_8w="mean", max_8w="max").reset_index()
out = (grid.merge(ml, on=["client_id", "ds"], how="left").merge(base, on="client_id", how="left")
           .merge(scheduled.rename(columns={"y": "scheduled"}), on=["client_id", "ds"], how="left"))
out["scheduled"] = out["scheduled"].fillna(0)
fc = out["forecast"].fillna(out["mean_8w"]).fillna(0)
out["metric_forecast"] = np.maximum(fc, out["scheduled"])
out["metric_lower"] = np.maximum(out["forecast_lower"].fillna(0), out["scheduled"])
out["metric_upper"] = np.maximum.reduce([out["forecast_upper"].fillna(out["max_8w"]).fillna(0), fc, out["scheduled"]])
out["method"] = np.where(out["forecast"].notna(), "lightgbm_global", np.where(out["mean_8w"].notna(), "baseline_mean_8w", "no_history"))
out = out.assign(ds=out.ds.dt.date, forecast_generated_at=pd.Timestamp.utcnow().tz_localize(None), mlflow_run_id=RUN_ID, model_uri=MODEL_URI)
out = out[["client_id", "ds", "metric_forecast", "metric_upper", "metric_lower", "method", "forecast_generated_at", "mlflow_run_id", "model_uri"]]

assert len(out) == H * len(clients) and out[["metric_forecast", "metric_lower", "metric_upper"]].notna().all().all()
assert ((out.metric_lower <= out.metric_forecast) & (out.metric_forecast <= out.metric_upper) & (out.metric_lower >= 0)).all()
(spark.createDataFrame(out).write.mode("overwrite").option("overwriteSchema", "true")
      .saveAsTable(f"{CATALOG}.gold.launch_forecasts_mlflow"))
display(spark.table(f"{CATALOG}.gold.launch_forecasts_mlflow").groupBy("client_id", "method").count())